In [ ]:
# https://github.com/LuGodoy/conversando-por-voz.git

In [ ]:
language = 'pt'

# 1. Gravação de Áudio Com Python (e Uma Pitada de JavaScript) 🎤

In [25]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=8):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

# 2. Reconhecimento de Fala com Whisper (OpenAI) 🧠

In [ ]:
!pip install git+https://github.com/openai/whisper.git -q

# 3. Transcrição de voz para texto 📝

In [26]:
import whisper

# Selecione o modelo do Whisper que melhor atenda às suas necessidades:
# https://github.com/openai/whisper#available-models-and-languages
model = whisper.load_model("small")

# Transcreve o audio gravado anteriormente.
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print(transcription)

 Qual o melhor serviço de investimento do Banco Baradesco?


# 4. Integração com a API do GEMINI 💬

In [27]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

In [28]:
gemini_model = genai.GenerativeModel('models/gemini-2.5-flash')

In [29]:
text = f"Atue como um especialista nos serviços do Banco Bradesco. Responda à pergunta do cliente de forma clara e objetiva, limitando-se a 4 linhas de texto. Pergunta: {transcription} Resposta:"

print(f"Conteúdo enviado ao Gemini: {text}")

Conteúdo enviado ao Gemini: Atue como um especialista nos serviços do Banco Bradesco. Responda à pergunta do cliente de forma clara e objetiva, limitando-se a 4 linhas de texto. Pergunta:  Qual o melhor serviço de investimento do Banco Baradesco? Resposta:


# Geração da resposta do Gemini em texto

In [30]:
import re

response = gemini_model.generate_content(text)
# Remove Markdown asterisks from the response text
cleaned_response_text = re.sub(r'\*', '', response.text)
print(cleaned_response_text)

Não há um único "melhor" serviço, pois o ideal depende do seu perfil, objetivos e tolerância a risco.

No Bradesco, oferecemos um portfólio completo, de Renda Fixa a Fundos de Investimento e Previdência.

Recomendamos consultar seu gerente ou um especialista Bradesco.

Eles farão uma análise para indicar a solução ideal para você.


# 5. Sintetizando a Resposta do Gemini Como Voz (gTTS) 🔊

In [31]:
gtts_object = gTTS(text=cleaned_response_text, lang=language, slow=False)
# Salva o áudio da resposta no arquivo especificado (pasta padrão do Google Colab)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

# Reproduz o áudio da resposta salvo no arquivo
display(Audio(response_audio, autoplay=True))